# 02 · Falcon 9 Landing Outcomes and Data Wrangling
**Historical IBM SpaceX capstone | Portfolio reconstruction**

This notebook consumes the **90-row historical IBM dataset** from Notebook 01, profiles the data, and creates a transparent binary landing label. The original coursework used positional indexes into `value_counts()` to select unsuccessful outcomes; that approach is brittle because the order of categories can change. Here the mapping is explicit, and unexpected labels cause an error.

**Attribution:** Based on the IBM Applied Data Science Capstone learning sequence. Original notebook preserved under `original-coursework/`. No causal inference is claimed.

## 1. Configuration and input

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
FIGURES = ROOT / 'images'
for folder in (RAW, PROCESSED, FIGURES): folder.mkdir(parents=True, exist_ok=True)
SOURCE_URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv'
RAW_CSV = RAW / 'ibm_spacex_dataset_part_1.csv'
def load_source():
    if RAW_CSV.exists():
        return pd.read_csv(RAW_CSV)
    import requests
    response = requests.get(SOURCE_URL, timeout=30)
    response.raise_for_status()
    RAW_CSV.write_bytes(response.content)
    return pd.read_csv(RAW_CSV)

In [2]:
source = PROCESSED / 'falcon9_launches_historical.csv'
if source.exists():
    df = pd.read_csv(source)
    print('Loaded Notebook 01 output:', source)
else:
    df = load_source()
    print('Loaded verified IBM historical source (run Notebook 01 to generate the standard output).')
assert not df.empty
required = {'Date','Outcome','Orbit','LaunchSite','PayloadMass','FlightNumber'}
if missing := required - set(df.columns): raise ValueError(f'Missing source columns: {sorted(missing)}')
df['Date'] = pd.to_datetime(df['Date'], errors='raise')
print('Shape:', df.shape)
display(df.head())

Loaded Notebook 01 output: c:\Users\kanwa\Downloads\New folder\notebooks\data\processed\falcon9_launches_historical.csv
Shape: (90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 2. Missingness and exploratory distributions

In [3]:
missing = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_pct': df.isna().mean().mul(100).round(1)})
display(missing.sort_values('missing_count', ascending=False))
for field in ['LaunchSite','Orbit','Outcome']:
    print(f'\n{field}')
    display(df[field].value_counts(dropna=False).rename('launches').to_frame())

,missing_count,missing_pct
LandingPad,26,28.9
FlightNumber,0,0.0
Date,0,0.0
PayloadMass,0,0.0
BoosterVersion,0,0.0
LaunchSite,0,0.0
Outcome,0,0.0
Flights,0,0.0
Orbit,0,0.0
GridFins,0,0.0



LaunchSite


,launches
LaunchSite,
CCAFS SLC 40,55
KSC LC 39A,22
VAFB SLC 4E,13



Orbit


,launches
Orbit,
GTO,27
ISS,21
VLEO,14
PO,9
LEO,7
SSO,5
MEO,3
HEO,1
ES-L1,1



Outcome


,launches
Outcome,
True ASDS,41
None None,19
True RTLS,14
False ASDS,6
True Ocean,5
False Ocean,2
None ASDS,2
False RTLS,1


## 3. Define a reproducible landing label

**Positive (`Class = 1`):** successful first-stage recovery, whether at a ground pad, drone ship, or ocean landing zone. **Negative (`Class = 0`):** failed recovery or no successful landing (including explicitly recorded `None None` and `None ASDS` in the original IBM teaching convention). An unrecognised outcome is **not silently labelled**; inspect and decide before continuing.

This is a **course-style recovery outcome**, not a mission-success outcome. The mapping should be reviewed if the source data or research question changes.

In [4]:
print('Observed outcomes:')
display(df['Outcome'].value_counts(dropna=False))

def classify_outcome(value):
    if pd.isna(value):
        return pd.NA
    label = str(value).strip()
    status = label.split(' ', 1)[0]
    if status == 'True': return 1
    if status in {'False', 'None'}: return 0
    raise ValueError(f'Unexpected landing outcome: {value!r}')

df['Class'] = pd.Series(df['Outcome'].map(classify_outcome), index=df.index, dtype='Int64')
if df['Class'].isna().any():
    raise ValueError('Unknown landing labels found; resolve before export')
print('Class distribution:')
display(df['Class'].value_counts().sort_index().rename('launches').to_frame())
print(f'Observed recovery success proportion: {df["Class"].mean():.1%}')

Observed outcomes:


Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

Class distribution:


,launches
Class,
0,30
1,60


Observed recovery success proportion: 66.7%


## 4. Validate and export

In [5]:
assert df['Class'].isin([0,1]).all()
assert not df['Date'].isna().any()
assert not df.duplicated().any(), 'Unexpected duplicate rows'
output = PROCESSED / 'falcon9_labeled_historical.csv'
export_df = df.copy()
export_df['Date'] = export_df['Date'].dt.strftime('%Y-%m-%d')
export_df.to_csv(output, index=False)
print('Exported:', output, 'shape:', export_df.shape)
print('Class 0:', int((df['Class']==0).sum()), '| Class 1:', int((df['Class']==1).sum()))

Exported: c:\Users\kanwa\Downloads\New folder\notebooks\data\processed\falcon9_labeled_historical.csv shape: (90, 18)
Class 0: 30 | Class 1: 60


## Limitations and next steps
- Class is based on the **historical IBM `Outcome` coding**, which can include non-attempts as zero; this is not a pure *attempted-landing success* metric.
- No payload-mass imputation is performed. Downstream modelling should fit imputers **only on training data**.
- Data is historical; avoid generalising directly to present-day SpaceX operations.
- Notebook 05 consumes `falcon9_labeled_historical.csv`.